# QCNN - Hilbert-space evolution and data visualization


## 1. Model and test data

Choose the model, dataset and completed training run to analyse: the defaults are `model17`, `fashion4_2` and seed `42`. Available datasets are `fashion4_1`, `fashion4_2`, `fashion4_3`, `mnist` and `organamnist`. Load the best parameters saved during training and prepare a random sample of **500 official test images per class** (2000 total) by default.

Set `N_PER_CLASS` to change this analysis sample size. `SAMPLE_SEED` controls image selection independently of the training seed; the selected images can differ from the test subset in `test.json`. This setting does not change the training split.


In [ ]:
# Import analysis libraries and locate the existing qcnn project packages.
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import torch
import pennylane as qml
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "training" / "train.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from a directory inside the qcnn project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets import get_dataset_config, get_test_loader
from models import get_model
from models.utilities.encoding import (
    compute_e1_gamma, e1_global_stats_4, e1_quad_means_8, patches_4x15,
)
from models.utilities.qcnn_model import OUTPUT_WIRES, POOL_STAGE1


In [ ]:
# Change these values to analyse another existing training run.
MODEL_NAME = "model17"
DATASET_NAME = "fashion4_2"  # Also: mnist, organamnist, fashion4_1, fashion4_3.
SEED = 42
N_PER_CLASS = 500  # Random test images per class; 2000 images in total.
SAMPLE_SEED = 123  # Reproducible image sampling, independent of the training seed.

DATA_DIR = PROJECT_ROOT / "datasets" / "data"
RUN_DIR = PROJECT_ROOT / "results" / MODEL_NAME / DATASET_NAME / f"seed{SEED}"
WEIGHTS_PATH = RUN_DIR / "best_state.pt"

# Batching limits working memory without changing the circuits or fidelity definition.
STATE_BATCH_SIZE = 100
FIDELITY_DEVICE = "cpu"
FIDELITY_BLOCK_4Q = 48
FIDELITY_BLOCK_2Q = 128

MDS_N_INIT = 3
MDS_MAX_ITER = 500
MDS_EPS = 1e-5

print("run             :", RUN_DIR)
print("trained weights :", WEIGHTS_PATH)
print("fidelity device :", FIDELITY_DEVICE)


In [ ]:
# Require a completed run: never substitute random parameters or the latest epoch.
for filename in ("config.json", "test.json", "best_state.pt"):
    if not (RUN_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing training artifact: {RUN_DIR / filename}")

config = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))
test_result = json.loads((RUN_DIR / "test.json").read_text(encoding="utf-8"))
dataset_config = get_dataset_config(DATASET_NAME)
for record in (config, test_result):
    expected = {"model": MODEL_NAME, "dataset": DATASET_NAME, "seed": SEED,
                **dataset_config}
    if any(record.get(key) != value for key, value in expected.items()):
        raise ValueError("Run metadata disagrees with the requested model/dataset/seed/classes.")

# Constructor parameters are placeholders; strict loading replaces every saved tensor.
with torch.random.fork_rng(devices=[]):
    model = get_model(MODEL_NAME).cpu()
if any(config.get(key) != value for key, value in model.model_config().items()):
    raise ValueError("The current model architecture differs from the saved training configuration.")
if test_result["architecture"] != model.architecture_name:
    raise ValueError("test.json refers to a different architecture.")

trained = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
if (
    trained["seed"] != SEED
    or trained["epoch"] != test_result["best_epoch"]
    or not np.isclose(trained["val_acc"], test_result["val_acc_best"], rtol=0, atol=1e-10)
    or not np.isclose(trained["val_loss"], test_result["best_val_loss"], rtol=0, atol=1e-10)
    or trained["n_params"] != test_result["n_params"]
    or model.n_trainable_params() != test_result["n_params"]
):
    raise ValueError("best_state.pt does not match the best state documented in test.json.")
model.load_state_dict(trained["model_state"], strict=True)
model.eval().requires_grad_(False)

CLASS_NAMES = dataset_config["class_names"]
N_QUBITS = model.n_qubits
POOL1_WIRES = [keep for _, keep in POOL_STAGE1]
FINAL_WIRES = list(OUTPUT_WIRES)
STAGE_NAMES = [
    f"After encoding ({N_QUBITS}q pure state)",
    f"After first pooling (reduced wires {POOL1_WIRES})",
    f"Before measurement (reduced wires {FINAL_WIRES})",
]

display(pd.DataFrame([{
    "model": MODEL_NAME, "dataset": DATASET_NAME, "seed": SEED,
    **model.model_config(), "best_epoch": trained["epoch"],
    "trainable_params": trained["n_params"], "saved_test_acc": test_result["test_acc"],
}]))


In [ ]:
# Load only official test images, with the preprocessing expected by the selected model.
if not isinstance(N_PER_CLASS, int) or N_PER_CLASS < 2:
    raise ValueError("N_PER_CLASS must be an integer >= 2 for within-class fidelity statistics.")

# Select balanced test samples using the training preprocessing;
# this analysis does not load the training or validation pools.
test_loader = get_test_loader(
    DATASET_NAME, data_dir=DATA_DIR, image_size=model.image_size,
    seed=SAMPLE_SEED, batch_size=STATE_BATCH_SIZE, test_per_class=N_PER_CLASS,
)
selected_indices = np.asarray(test_loader.dataset.indices, dtype=np.int64)
test_batches = list(test_loader)
images = torch.cat([x for x, _ in test_batches], dim=0)
labels = torch.cat([y for _, y in test_batches], dim=0).numpy()
del test_batches

print("images:", tuple(images.shape))
print("class counts:", {name: int((labels == c).sum()) for c, name in enumerate(CLASS_NAMES)})
print("sampling seed:", SAMPLE_SEED)


## 2. Quantum states at the three stages

For each selected test image, run the QCNN with the trained parameters loaded from `best_state.pt` and compute its quantum state at three stages:

1. **After encoding:** the full pure state representing the image, before convolution and pooling.
2. **After the first pooling:** the reduced density matrix of the four surviving qubits, `[0, 2, 4, 6]`.
3. **Before measurement:** the reduced density matrix of the two output qubits, `[0, 4]`, after the second convolution and pooling.

A **reduced density matrix** describes only the qubits retained at that stage. It is obtained by taking the **partial trace** over the discarded qubits and, for E1, the ancilla.

The diagonal of the final density matrix gives the four class probabilities: `P(00)`, `P(01)`, `P(10)` and `P(11)`, in the dataset's class order. Predict the class with the highest probability, compute accuracy on the selected images, and compare it with the accuracy saved in `test.json`.

These three sets of states are then used for the fidelity comparisons and MDS plots in the following sections.


In [ ]:
def model_circuit_tape(model, x_flat):
    """Trace the trained QNode with the feature preparation used by its forward pass."""
    tail = (model.theta_fine, model.theta_pool1, model.theta_coarse, model.theta_pool2)
    if model.encoding_name == "reupload":
        args = (patches_4x15(x_flat), model.theta_reupload, model.weight_reupload, *tail)
    elif model.encoding_name == "e1":
        args = (e1_quad_means_8(x_flat), compute_e1_gamma(e1_global_stats_4(x_flat)),
                model.a_embed, model.c_embed, *tail)
    elif model.encoding_name == "e3":
        args = (x_flat, *tail)
    else:
        raise ValueError(f"Unsupported encoding: {model.encoding_name}")
    return model.qnode.construct(args, {})


def circuit_stage_lengths(model):
    """Count operations by invoking the existing stage methods, including empty pooling."""
    stages = (
        (model._apply_fine, model.theta_fine),
        (model._apply_pool1, model.theta_pool1),
        (model._apply_coarse, model.theta_coarse),
        (model._apply_pool2, model.theta_pool2),
    )
    return [len(qml.tape.make_qscript(method)(theta).operations) for method, theta in stages]


def evaluate_states_batched(model, images, batch_size=100):
    """Simulate the encoding, first-pooling and final prefixes of the real trained circuit."""
    n_fine, n_pool1, n_coarse, n_pool2 = circuit_stage_lengths(model)
    outputs = [[], [], []]
    with torch.no_grad():
        for start in tqdm(range(0, len(images), batch_size), desc="three circuit stages"):
            tape = model_circuit_tape(model, images[start:start + batch_size])
            encoding_end = len(tape.operations) - n_fine - n_pool1 - n_coarse - n_pool2
            if encoding_end <= 0:
                raise RuntimeError("Cannot identify the encoding in the current model circuit.")
            stops = (encoding_end, encoding_end + n_fine + n_pool1, len(tape.operations))
            # Only the terminal measurement changes; all operations and parameters are retained.
            stage_tapes = tuple(qml.tape.QuantumScript(tape.operations[:stop], [qml.state()])
                                for stop in stops)
            states = qml.execute(stage_tapes, model.dev, interface="torch", diff_method=None)
            for stage, state in zip(outputs, states):
                stage.append(state.detach().cpu().numpy())
    return tuple(np.concatenate(stage, axis=0).astype(np.complex64) for stage in outputs)


states_enc, states_pool1_full, states_final_full = evaluate_states_batched(
    model, images, batch_size=STATE_BATCH_SIZE,
)
for name, states in zip(STAGE_NAMES, (states_enc, states_pool1_full, states_final_full)):
    print(f"{name}: {states.shape}")
    if not np.allclose(np.sum(np.abs(states) ** 2, axis=1), 1.0, atol=1e-5):
        raise RuntimeError(f"State normalization failed: {name}")


In [ ]:
# ------------------------------------------------------------
# REDUCED STATES ON THE QUBITS THAT SURVIVE THE POOLING
# ------------------------------------------------------------
def reduced_density_matrices_from_pure_states(
    states,
    keep_wires,
    n_qubits=8,
):
    states = np.asarray(states)
    batch = states.shape[0]

    psi = states.reshape(
        (batch,) + (2,) * n_qubits
    )

    keep_wires = list(keep_wires)

    env_wires = [
        w
        for w in range(n_qubits)
        if w not in keep_wires
    ]

    perm = (
        [0]
        + [w + 1 for w in keep_wires]
        + [w + 1 for w in env_wires]
    )

    psi = np.transpose(
        psi,
        axes=perm,
    )

    d_keep = 2 ** len(keep_wires)
    d_env = 2 ** len(env_wires)

    psi = psi.reshape(
        batch,
        d_keep,
        d_env,
    )

    rho = np.einsum(
        "bik,bjk->bij",
        psi,
        psi.conj(),
        optimize=True,
    )

    rho = 0.5 * (
        rho
        + np.swapaxes(
            rho.conj(),
            -1,
            -2,
        )
    )

    return rho.astype(np.complex64)



# Trace out discarded data wires and, for E1, the encoding ancilla.
rho_pool1 = reduced_density_matrices_from_pure_states(
    states_pool1_full, keep_wires=POOL1_WIRES, n_qubits=N_QUBITS,
)
rho_final = reduced_density_matrices_from_pure_states(
    states_final_full, keep_wires=FINAL_WIRES, n_qubits=N_QUBITS,
)
for name, rho in (("pool1", rho_pool1), ("final", rho_final)):
    trace_error = np.max(np.abs(np.trace(rho, axis1=1, axis2=2) - 1))
    print(f"rho {name}: {rho.shape}, max trace error: {trace_error:.3e}")
    if trace_error > 1e-5:
        raise RuntimeError(f"Reduced-state normalization failed: {name}")

# Diagonal entries follow the model's output-wire order: P(00), P(01), P(10), P(11).
final_probs = np.real(np.diagonal(rho_final, axis1=1, axis2=2))


# Use the final-state probabilities already computed above; no extra inference is needed.
predictions = final_probs.argmax(axis=1)
reconstructed_test_acc = float(np.mean(predictions == labels))
saved_test_acc = float(test_result["test_acc"])
accuracy_difference = reconstructed_test_acc - saved_test_acc

# Compare accuracies descriptively: the sampled images can differ from the saved test subset.
print(f"Accuracy on sampled test images ({len(labels)}): {reconstructed_test_acc:.2%}")
print(f"Recorded test accuracy ({test_result['test_size']} images): {saved_test_acc:.2%}")
print(f"Accuracy difference (sampled - recorded): {100 * accuracy_difference:+.2f} percentage points")


## 3. Fidelity analysis

Measure how similar the quantum states of different images are using **fidelity**, from 0 (fully distinguishable states) to 1 (identical states). At each of the three circuit stages, compare the average fidelity **within the same class** and **between different classes**, including each class against all the others. Compare these values across stages to see how class separation changes along the circuit.

The comparison table reports `within_minus_between`: the mean within-class fidelity minus the mean between-class fidelity.


In [ ]:
# ------------------------------------------------------------
# FIDELITY HELPERS
#
# Pure:
#   F = |<psi_i|psi_j>|^2
#
# Mixed:
#   squared Uhlmann fidelity
#   F(rho,sigma) =
#   [Tr sqrt(sqrt(rho) sigma sqrt(rho))]^2
# ------------------------------------------------------------
def pure_fidelity_matrix(states):
    s = np.asarray(
        states,
        dtype=np.complex64,
    )

    overlaps = s.conj() @ s.T

    F = np.abs(overlaps) ** 2

    F = np.clip(
        F.real,
        0.0,
        1.0,
    ).astype(np.float32)

    np.fill_diagonal(
        F,
        1.0,
    )

    return F


def _sqrt_psd_batch_torch(rhos_t):
    vals, vecs = torch.linalg.eigh(rhos_t)

    vals = torch.clamp(
        vals.real,
        min=0.0,
    )

    weighted_vecs = (
        vecs
        * torch.sqrt(vals).unsqueeze(-2)
    )

    return weighted_vecs @ vecs.mH


def mixed_fidelity_matrix(
    rhos,
    block_size,
    device=FIDELITY_DEVICE,
):
    """
    Evaluate squared Uhlmann fidelity numerically for every state pair.

    No cache and no resume logic.
    Block processing is used only to keep memory usage controlled.
    """
    n, d, _ = rhos.shape

    rhos_t = torch.as_tensor(
        rhos,
        dtype=torch.complex64,
        device=device,
    )

    with torch.inference_mode():
        sqrt_t = _sqrt_psd_batch_torch(
            rhos_t
        )

    F = np.empty(
        (n, n),
        dtype=np.float32,
    )

    t0 = time.time()

    with torch.inference_mode():
        for i0 in tqdm(
            range(0, n, block_size),
            desc=f"mixed fidelity d={d}",
        ):
            i1 = min(
                i0 + block_size,
                n,
            )

            sqrt_i = sqrt_t[i0:i1]

            for j0 in range(
                i0,
                n,
                block_size,
            ):
                j1 = min(
                    j0 + block_size,
                    n,
                )

                rho_j = rhos_t[j0:j1]

                m = torch.matmul(
                    sqrt_i[:, None, :, :],
                    rho_j[None, :, :, :],
                )

                m = torch.matmul(
                    m,
                    sqrt_i[:, None, :, :],
                )

                m = 0.5 * (
                    m + m.mH
                )

                eigvals = torch.linalg.eigvalsh(
                    m
                ).real

                eigvals = torch.clamp(
                    eigvals,
                    min=0.0,
                )

                root_fid = torch.sqrt(
                    eigvals
                ).sum(dim=-1)

                fid = torch.clamp(
                    root_fid.square(),
                    0.0,
                    1.0,
                )

                block = (
                    fid.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                F[i0:i1, j0:j1] = block
                F[j0:j1, i0:i1] = block.T

    # Remove eigensolver roundoff asymmetry, including within diagonal blocks.
    F = 0.5 * (F + F.T)

    F = np.clip(
        F,
        0.0,
        1.0,
    )

    np.fill_diagonal(
        F,
        1.0,
    )

    print(
        f"completed mixed fidelity d={d} "
        f"in {(time.time() - t0)/60:.1f} min"
    )

    return F


def fidelity_report(
    F,
    labels,
    class_names,
    stage_name,
):
    """Collect fidelity statistics for the stage comparison tables."""
    labels = np.asarray(labels)
    n = len(labels)

    same = (
        labels[:, None]
        == labels[None, :]
    )

    upper = np.triu(
        np.ones(
            (n, n),
            dtype=bool,
        ),
        k=1,
    )

    within_vals = F[
        same & upper
    ]

    between_vals = F[
        (~same) & upper
    ]

    mean_within = float(
        within_vals.mean()
    )

    mean_between = float(
        between_vals.mean()
    )

    within_per_class = {}
    between_per_pair = {}
    one_vs_all = {}

    for c, name in enumerate(class_names):
        idx = np.where(
            labels == c
        )[0]

        block = F[
            np.ix_(idx, idx)
        ]

        vals = block[
            np.triu_indices(
                len(idx),
                k=1,
            )
        ]

        m = float(
            vals.mean()
        )

        within_per_class[name] = m

    for c1 in range(len(class_names)):
        for c2 in range(
            c1 + 1,
            len(class_names),
        ):
            idx1 = np.where(
                labels == c1
            )[0]

            idx2 = np.where(
                labels == c2
            )[0]

            vals = F[
                np.ix_(idx1, idx2)
            ].reshape(-1)

            m = float(
                vals.mean()
            )

            key = (
                f"{class_names[c1]} "
                f"vs {class_names[c2]}"
            )

            between_per_pair[key] = m

    for c, name in enumerate(class_names):
        pair_means = []

        for other_c in range(
            len(class_names)
        ):
            if c == other_c:
                continue

            a, b = sorted(
                (c, other_c)
            )

            key = (
                f"{class_names[a]} "
                f"vs {class_names[b]}"
            )

            pair_means.append(
                between_per_pair[key]
            )

        m = float(
            np.mean(pair_means)
        )

        one_vs_all[name] = m

    return {
        "stage": stage_name,
        "mean_within": mean_within,
        "mean_between": mean_between,
        "within_per_class": within_per_class,
        "between_per_pair": between_per_pair,
        "one_vs_all": one_vs_all,
    }


In [ ]:
# A: full pure state after the selected model's encoding (including the E1 ancilla).
F_enc = pure_fidelity_matrix(states_enc)
report_enc = fidelity_report(F_enc, labels, CLASS_NAMES, STAGE_NAMES[0])


In [ ]:
# B: mixed state of the four surviving data wires after the first pooling.
F_pool1 = mixed_fidelity_matrix(
    rho_pool1, block_size=FIDELITY_BLOCK_4Q, device=FIDELITY_DEVICE,
)
report_pool1 = fidelity_report(F_pool1, labels, CLASS_NAMES, STAGE_NAMES[1])


In [ ]:
# C: mixed state of the two output wires immediately before class readout.
F_final = mixed_fidelity_matrix(
    rho_final, block_size=FIDELITY_BLOCK_2Q, device=FIDELITY_DEVICE,
)
report_final = fidelity_report(F_final, labels, CLASS_NAMES, STAGE_NAMES[2])


In [ ]:
# ------------------------------------------------------------
# STEP-BY-STEP NUMERICAL COMPARISON
# ------------------------------------------------------------
reports = [
    report_enc,
    report_pool1,
    report_final,
]

stage_short = [
    f"After encoding ({N_QUBITS}q)",
    "After pool1 (4q)",
    "Before measurement (2q)",
]

overall_df = pd.DataFrame([
    {
        "stage": stage_short[i],
        "mean_within": r["mean_within"],
        "mean_between": r["mean_between"],
        "within_minus_between": r["mean_within"] - r["mean_between"],
    }
    for i, r in enumerate(reports)
])

within_df = pd.DataFrame(
    {
        stage_short[i]:
        r["within_per_class"]
        for i, r in enumerate(reports)
    }
)

within_df.index.name = "class"

ova_df = pd.DataFrame(
    {
        stage_short[i]:
        r["one_vs_all"]
        for i, r in enumerate(reports)
    }
)

ova_df.index.name = "class"

pairwise_df = pd.DataFrame(
    {
        stage_short[i]:
        r["between_per_pair"]
        for i, r in enumerate(reports)
    }
)

pairwise_df.index.name = "class pair"

print("OVERALL EVOLUTION")
display(overall_df)

print("WITHIN-CLASS FIDELITY EVOLUTION")
display(within_df)

print("ONE-VS-ALL FIDELITY EVOLUTION")
display(ova_df)

print("PAIRWISE BETWEEN-CLASS FIDELITY EVOLUTION")
display(pairwise_df)


## 4. MDS visualization

Create a **two-dimensional map** of the images for each of the three circuit stages. Each point represents an image, and its colour identifies the class. MDS places images with similar quantum states close together, approximately preserving the similarities measured by fidelity.

**Reading normalized stress (Stress-1).** Stress measures how much the 2D map distorts the original quantum-state distances. Lower is better. The following values are rough reference points:

| Normalized stress | Quality of the 2D representation |
|---|---|
| 0 | Perfect preservation of distances |
| Around 0.025 | Excellent |
| Around 0.05 | Good |
| Around 0.10 | Fair |
| Around 0.20 or higher | Poor; interpret distances in the plot cautiously |


In [ ]:
# ------------------------------------------------------------
# MDS FOR ALL THREE SNAPSHOTS
#
# D_ij = sqrt(1 - F_ij)
# ------------------------------------------------------------
# Convert fidelity to distance for the MDS maps.
def fidelity_to_distance(F):
    distances = np.sqrt(np.clip(1.0 - F, 0.0, 1.0)).astype(np.float32)
    # Remove numerical asymmetry and enforce zero distance from each image to itself.
    distances = 0.5 * (distances + distances.T)
    np.fill_diagonal(distances, 0.0)
    return distances


def run_mds_from_fidelity(
    F,
    name,
):
    # Derive the input distances directly from the fidelity matrix.
    D = fidelity_to_distance(F)

    print(
        f"running MDS: {name} | "
        f"n={len(D)}"
    )

    t0 = time.time()

    mds = MDS(
        n_components=2,
        metric_mds=True,
        metric="precomputed",
        init="random",
        normalized_stress=True,
        random_state=SEED,
        n_init=MDS_N_INIT,
        max_iter=MDS_MAX_ITER,
        eps=MDS_EPS,
        verbose=0,
    )

    Z = mds.fit_transform(D)

    elapsed = time.time() - t0

    print(
        f"completed {name}: "
        f"normalized stress={mds.stress_:.6f} | "
        f"{elapsed/60:.1f} min"
    )

    return Z, float(mds.stress_)


Z_enc, stress_enc = run_mds_from_fidelity(
    F_enc,
    STAGE_NAMES[0],
)

Z_pool1, stress_pool1 = run_mds_from_fidelity(
    F_pool1,
    STAGE_NAMES[1],
)

Z_final, stress_final = run_mds_from_fidelity(
    F_final,
    STAGE_NAMES[2],
)

print()
print("NORMALIZED MDS STRESS (lower is better)")
print(
    f"after encoding      : "
    f"{stress_enc:.6f}"
)
print(
    f"after first pooling : "
    f"{stress_pool1:.6f}"
)
print(
    f"before measurement  : "
    f"{stress_final:.6f}"
)


In [ ]:
# Plot the three stages with labels from the selected dataset.
def plot_mds(
    Z,
    title,
):
    plt.figure(
        figsize=(8, 7)
    )

    for c, name in enumerate(CLASS_NAMES):
        mask = labels == c

        plt.scatter(
            Z[mask, 0],
            Z[mask, 1],
            s=14,
            alpha=0.55,
            label=name,
        )

    plt.xlabel(
        "MDS dimension 1"
    )

    plt.ylabel(
        "MDS dimension 2"
    )

    plt.title(
        title
    )

    plt.legend(
        markerscale=1.8
    )

    plt.tight_layout()
    plt.show()


for coordinates, stage_name in zip((Z_enc, Z_pool1, Z_final), STAGE_NAMES):
    plot_mds(
        coordinates,
        f"{stage_name}\n"
        f"selected {N_PER_CLASS} images per class",
    )
